# Partition uncertainties in projections of impacts, accounting for regression uncertainty

For the balanced panel of BCD-ME runs, MC draws over the VCV matrix of regression results

In [2]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [3]:
from distributed import Client

# Start dask client 
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39401 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/39401/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/39401/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34381,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/39401/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41267,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/34323/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:34453,


In [4]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

nsamples = 1000
rng = np.random.default_rng()

max_gwl = 3
nruns = 13

## Calculate uncertainty with MC draws of regression uncertainty

In [5]:
overwrite = False
for proc in ['mortality','gdppc']:#params_proc:
    # Get filename of output uncertainty
    output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_wregunc.zarr'

    if (not overwrite) and os.path.exists(output_fn):
        print(output_fn+' exists, skipped!')
        continue
    
    # Get filenames of aggregated climate predictors
    fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/{params_proc[proc]['varname']}_*_{params_proc[proc]['geo_typ']}*'))
    if params_proc[proc]['popwght']:
        fns = [fn for fn in fns if re.search(r'\_popwght\.zarr',fn)]
    else:
        fns = [fn for fn in fns if not re.search(r'\_popwght\.zarr',fn)]
    fns = {re.split(r'\_',re.split(r'\/',fn)[-1])[2]:fn for fn in fns}
    
    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc()
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality()
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll()
    else:
        raise KeyError()

    # Get MC samples of coefficients
    coeffs_mvn = xr.apply_ufunc(rng.multivariate_normal,
                               coeffs.coeffs,
                               vcv.vcv,
                               input_core_dims = [['pred'],['pred_1','pred_2']],
                               output_core_dims = [['sidx','pred']],
                               kwargs = {'size':nsamples},
                                       dask = 'parallelized',
                                       dask_gufunc_kwargs = {'output_sizes':{'sidx':nsamples}},
                               vectorize=True)

    dss = []
    for mod in tqdm(fns):
    #for mod in ['ACCESS-ESM1-5']:
        tmp_fn = f'{dir_list['proc']}tmp/d{proc}_regMCs_{mod}TMP.zarr'
        if overwrite or (not os.path.exists(tmp_fn)):
    
            # Load aggregated climate predictor
            ds = xr.open_zarr(fns[mod],consolidated=False,decode_timedelta=False)
            if 'model' not in ds:
                ds = ds.assign_coords({'model':(('idv'),[mod]*ds.sizes['idv'])})
        
            # Isolate predictor variables
            if params_proc[proc]['varname'] == 'tassumpoly':
                ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
                ds = ds.rename({'degree':'pred'})
            elif params_proc[proc]['varname'] == 'tasmaxbinF':
                ds['binF'] = [f'tasmaxbinF_{k}' for k in ds.binF.values]
                ds = ds.rename({'binF':'pred'})
            else:
                raise KeyError()

            ds = ds.chunk({'pred':-1,'gwl':1,'proj_base':-1,'poly_idx':20})
    
            # Calculate impact from linear regression model
            ds[proc] = (coeffs_mvn*ds[params_proc[proc]['varname']]).sum('pred',skipna=False)
            # Calculate change in impact vs. GWL0.61
            ds['d'+proc] = ds[proc] - ds[proc].sel(gwl=0.61)
    
            # Add max gwl to coordinates, for referece
            ds = ds.assign_coords({'maxgwl':(('idv'),ds.gwl.isel(gwl=np.array([g if g != 0 else -1 
                                                              for g in (np.isnan(ds[params_proc[proc]['varname']]).
                                                                        argmax('gwl').
                                                                        max([d for d in ds[params_proc[proc]['varname']].sizes if d not in ['gwl','idv']])).
                                                                        load()])).values)})
    
            # Limit to just impact variable
            ds = ds[['d'+proc]]
    
            # Limit to just the balanced panel
            if ds.isel(idv = ds.maxgwl>=max_gwl).sizes['idv'] < nruns:
                print(f'{mod} does not have enough runs for desired panel size, skipping.')
                continue
                
            else:
                ds = ds.isel(idv = ds.maxgwl>=max_gwl).isel(idv=slice(0, nruns))
    
            # Rechunk for uncertainty partitioning
            ds = ds.chunk({'poly_idx':20,'idv':-1,'sidx':-1,'proj_base':-1,'gwl':1})
    
            # Save temporary file
            ds['exprun'] = ds['experiment']+ds['run']
            ds = ds.set_index(idv = ['model','exprun'])
            ds = ds.reset_index('idv')
            ds.attrs = {'SOURCE':'4b_partition_uncertainty_wregunc.ipynb',
                         'DESCRIPTION':f'{proc} projections, calculated using {nsamples} MC draws, rechunked in preparation for uncertainty partitioning. Limited to the 13 runs / 9 mods "balanced" panel because of size reasons.'}

            utility_save(ds,tmp_fn,zarr_mode='w')


    #------------------------------------------------------
    # Uncertainty partitioning! ---------------------------
    #------------------------------------------------------
    # Load temp files
    dss = xr.concat([xr.open_zarr(f'{dir_list['proc']}tmp/d{proc}_regMCs_{mod}TMP.zarr')
       for mod in fns if os.path.exists(f'{dir_list['proc']}tmp/d{proc}_regMCs_{mod}TMP.zarr')],dim='idv',join='outer')

    dss = dss.set_index(idv = ['model','exprun'])
    
    dss = dss.chunk({'idv':-1,'proj_base':-1,'sidx':-1,'poly_idx':20,'gwl':1})


    uncerts_all = []
    #---------------------------
    # Uncertainty partitioning over a balanced panel up to GWL 3
    #---------------------------
    dss_tmp = dss.unstack('idv')['d'+proc]
    dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
    
    with warnings.catch_warnings():
        # Ignoring chunk increase warnings and divide by 0s in uncertainty 
        # partitioning (happens over ocean, e.g.)
        warnings.filterwarnings('ignore')
        uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()        
    uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
    
    uncerts_all.append(uncerts)
    
    #---------------------------
    # Uncertainty partitioning over a balanced panel up to GWL 3
    # without MERRA2
    #---------------------------
    dss_tmp = dss.sel(proj_base = [pb for pb in dss_tmp.proj_base.values if pb != 'MERRA2'])
    dss_tmp = dss_tmp.unstack('idv')['d'+proc]
    dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
    
    with warnings.catch_warnings():
        # Ignoring chunk increase warnings and divide by 0s in uncertainty 
        # partitioning (happens over ocean, e.g.)
        warnings.filterwarnings('ignore')
        uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
    
    uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods_noMERRA2']})
    
    uncerts_all.append(uncerts)
    
    #---------------------------
    # Concat and export
    #---------------------------
    uncerts_all = xr.concat(uncerts_all,dim='sample')

    uncerts_all.attrs = {'SOURCE':'4b_partition_uncertainty_wregunc.ipynb',
                         'DESCRIPTION':f'Uncertainty partitioning with {nsamples} MC draws for regression uncertainty'}


    utility_save(uncerts_all,output_fn)


    # Delete temp files


    

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality_wregunc.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_gdppc_wregunc.zarr exists, skipped!


## Get key quantiles across MC draws for plotting

In [35]:
for proc in ['mortality','gdppc']:
    output_fn = f'{dir_list['proc']}uncert_parts/regunc_qs_{proc}.zarr'

    if not os.path.exists(output_fn):
    
        # Get filenames of aggregated climate predictors
        fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/{params_proc[proc]['varname']}_*_{params_proc[proc]['geo_typ']}*'))
        if params_proc[proc]['popwght']:
            fns = [fn for fn in fns if re.search(r'\_popwght\.zarr',fn)]
        else:
            fns = [fn for fn in fns if not re.search(r'\_popwght\.zarr',fn)]
        fns = {re.split(r'\_',re.split(r'\/',fn)[-1])[2]:fn for fn in fns}
        
        # Load temp files
        dss = xr.concat([xr.open_zarr(f'{dir_list['proc']}tmp/d{proc}_regMCs_{mod}TMP.zarr')
           for mod in fns if os.path.exists(f'{dir_list['proc']}tmp/d{proc}_regMCs_{mod}TMP.zarr')],dim='idv',join='outer')
        
        dss = dss.set_index(idv = ['model','exprun'])

        # Rechunk to make quantile calculations possible
        dss = dss.chunk({'sidx':-1})

        # Calculate quantiles
        qlist = [0.025,0.1,0.25,0.5,0.75,0.9,0.975]
        qs = dss.quantile(qlist,dim='sidx')

        qs.attrs['SOURCE'] = '4b_partition_uncertainty.ipynb'
        qs.attrs['DESCRIPTION'] = 'Select quantiles of MC draws of regression uncertainty'

        utility_save(qs.reset_index('idv'),output_fn,zarr_mode = 'w')
    else:
        print(output_fn+' exists, skipped!')


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/regunc_qs_mortality.zarr exists, skipped!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=13, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change wit

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/regunc_qs_gdppc.zarr saved!
